**ATTENTION AND TRANSFORMERS**

**TRANSFORMER ARCHITECTURE FROM NUMPY**

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
x = np.random.randn(5,4)
x

array([[ 1.23844169, -0.15213365,  1.41072679, -0.88809966],
       [ 0.84311967, -0.10673557,  0.94155436,  0.40090893],
       [-2.08893555, -0.47162875, -1.05265116,  0.21601135],
       [ 1.14875786, -0.41782695, -0.39162539,  1.20833658],
       [-0.38511935,  0.29486193, -0.37199495, -0.78199555]])

In [ ]:
wq = np.random.rand(4,4)
wk = np.random.rand(4,4)
wv = np.random.rand(4,4)

In [ ]:
Q = np.dot(x,wq)
K = np.dot(x,wk)
V = np.dot(x,wv)

In [ ]:
raw_score = (Q @ K.T)/np.sqrt(4)

In [ ]:
raw_score.shape

(5, 5)

In [ ]:
softmax = np.exp(raw_score) / np.sum(np.exp(raw_score),axis=1,keepdims=True)

In [ ]:
output = np.dot(softmax,V)
output.shape

(5, 4)

In [ ]:
print(np.sum(softmax, axis=1))

[1. 1. 1. 1. 1.]


**ATTENTION AND TRANSFORMER APPLIED TO JPMC AND S&P500 DATA**

**PREPARING THE DATA**

In [ ]:
jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')

/tmp/ipykernel_709/4128732890.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_709/4128732890.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed


**EXPLORATORY DATA ANALYSIS**

In [ ]:
jpmc.isnull().sum().sum()

np.int64(0)

In [ ]:
sp500.isnull().sum().sum()

np.int64(0)

In [ ]:
jpmc.duplicated().sum()

np.int64(0)

In [ ]:
sp500.duplicated().sum()

np.int64(0)

In [ ]:
jpmc.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, JPM)   3774 non-null   float64
 1   (High, JPM)    3774 non-null   float64
 2   (Low, JPM)     3774 non-null   float64
 3   (Open, JPM)    3774 non-null   float64
 4   (Volume, JPM)  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


In [ ]:
sp500.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (Close, ^GSPC)   3774 non-null   float64
 1   (High, ^GSPC)    3774 non-null   float64
 2   (Low, ^GSPC)     3774 non-null   float64
 3   (Open, ^GSPC)    3774 non-null   float64
 4   (Volume, ^GSPC)  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


**FEATURE ENGINEERING**

In [ ]:
#Daily returns
jpmc['returns'] = jpmc['Close']['JPM'].pct_change(fill_method=None)

In [ ]:
#volume ratio
volume = jpmc['Volume']['JPM']
jpmc['volume ratio'] = (volume/ volume.rolling(window=20).mean()).shift(1)

In [ ]:
#20-day rolling return
jpmc['20-day rolling return'] = jpmc['returns'].rolling(window=20).mean().shift(1)

In [ ]:
delta = jpmc['Close']['JPM'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
RSI = 100 - (100 / (1 + gain/loss))
jpmc['RSI'] = RSI.shift(1)

In [ ]:
sp500['SP500 returns'] = sp500['Close']['^GSPC'].pct_change(fill_method=None)

In [ ]:
sp500_return_dataframe = pd.DataFrame({
    'Date': sp500.index,
    'SP500 returns': sp500['SP500 returns']
})
sp500_return_dataframe.set_index('Date', inplace=True)

In [ ]:
jpmc_clean = pd.DataFrame({
    "returns": jpmc['returns'],
    "Volume Ratio": jpmc['volume ratio'],
    "Rolling Returns": jpmc['20-day rolling return'],
    "RSI": jpmc['RSI']
})

In [ ]:
jpmc_clean = jpmc_clean.join(sp500_return_dataframe,how='inner')

In [ ]:
jpmc_clean['target'] = (jpmc_clean['returns'] > 0).astype(int).shift(-1)

In [ ]:
jpmc_clean.dropna(inplace=True)
jpmc_clean

,returns,Volume Ratio,Rolling Returns,RSI,SP500 returns,target
Date,,,,,,
2010-02-03,-0.006412,0.853654,-0.002507,36.612068,-0.005474,0.0
2010-02-04,-0.048151,0.696517,-0.003796,31.106896,-0.031141,0.0
2010-02-05,-0.001304,1.037733,-0.006478,23.539209,0.002897,0.0
2010-02-08,-0.015666,1.327887,-0.007534,25.589840,-0.008863,1.0
2010-02-09,0.018303,1.007097,-0.008194,25.133688,0.013040,1.0
...,...,...,...,...,...,...
2024-12-23,0.003325,3.523394,-0.001417,36.638914,0.007287,1.0
2024-12-24,0.016444,0.934839,-0.002024,39.867639,0.011043,1.0
2024-12-26,0.003425,0.419782,-0.001552,48.407832,-0.000406,0.0


In [ ]:
jpmc_clean.shape

(3752, 6)

In [ ]:
jpmc_clean.columns

Index(['returns', 'Volume Ratio', 'Rolling Returns', 'RSI', 'SP500 returns',
       'target'],
      dtype='object')

**SEQUENCES AND SPLITTING DATA**

In [ ]:
features = jpmc_clean.drop('target',axis=1)
target = jpmc_clean['target']

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [ ]:
target = target.to_numpy()

In [ ]:
lookback = 20
x,y = [],[]

for i in range (len(features_scaled) - lookback):
  x.append(features_scaled[i:i+lookback])
  y.append(target[i+lookback])

In [ ]:
x = np.array(x)
y = np.array(y)

In [ ]:
x.shape

(3732, 20, 5)

In [ ]:
y.shape

(3732,)

In [ ]:
x_train = x[:int(0.8*len(x))]
y_train = y[:int(0.8*len(y))]
x_test = x[int(0.8*len(x)):]
y_test = y[int(0.8*len(y)):]

In [ ]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(2985, 20, 5)
(2985,)
(747, 20, 5)
(747,)


**CREATING TRANSFORMER MODEL**

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class Transformer(nn.Module):
  def __init__(self,input_size,hidden_size):
    super().__init__()
    self.transformer_layer = nn.TransformerEncoderLayer(d_model = 32,nhead=4,batch_first = True)
    self.transformer = nn.TransformerEncoder(encoder_layer=self.transformer_layer,num_layers=2)
    self.input_proj = nn.Linear(input_size, 32)
    self.fc = nn.Linear(32, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,x):

    x = self.input_proj(x)
    output=self.transformer(x)
    out = output[:,-1,:]
    out = self.fc(out)
    return self.sigmoid(out)


In [ ]:
model = Transformer(5, 32)
print(model)

Transformer(
  (transformer_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
    )
    (linear1): Linear(in_features=32, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=2048, out_features=32, bias=True)
    (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2):

**TRAINING DATA PREPARATION**

In [ ]:
x_train_t = torch.tensor(x_train,dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32)
y_train_t = y_train_t.reshape(-1,1)

x_test_t = torch.tensor(x_test,dtype=torch.float32)
y_test_t = torch.tensor(y_test,dtype=torch.float32)
y_test_t = y_test_t.reshape(-1,1)

**MODEL TRAINING AND TESTING (WITHOUT POSITIONAL ENCODING)**

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs = 100
batch_size = 32

for epoch in range(epochs):
    model.train()
    for i in range(0, len(x_train_t), batch_size):
        x_batch = x_train_t[i:i+batch_size]
        y_batch = y_train_t[i:i+batch_size]
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0:
      print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

Epoch 0, Loss: 0.7078
Epoch 10, Loss: 0.6940
Epoch 20, Loss: 0.6823
Epoch 30, Loss: 0.5538
Epoch 40, Loss: 0.4402
Epoch 50, Loss: 0.2561
Epoch 60, Loss: 0.2142
Epoch 70, Loss: 0.1516
Epoch 80, Loss: 0.1340
Epoch 90, Loss: 0.1168


In [ ]:
model.eval()

with torch.no_grad():
  y_pred_train = model(x_train_t)
  y_pred_train_class = (y_pred_train > 0.5).float()
  train_accuracy = (y_pred_train_class == y_train_t).float().mean()
  print(f'Train Accuracy: {train_accuracy.item():.4f}')

Train Accuracy: 0.9012


In [ ]:
model.eval()

with torch.no_grad():
  y_pred_test = model(x_test_t)
  y_pred_test_class = (y_pred_test > 0.5).float()
  test_accuracy = (y_pred_test_class == y_test_t).float().mean()
  print(f'Train Accuracy: {test_accuracy.item():.4f}')

Train Accuracy: 0.5154


**MODEL TRAINING AND TESTING (WITH POSITIONAL ENCODING)**

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=20):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)  # shape (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [ ]:
class Transformer(nn.Module):
  def __init__(self,input_size,hidden_size):
    super().__init__()
    self.transformer_layer = nn.TransformerEncoderLayer(d_model = 32,nhead=4,batch_first = True)
    self.transformer = nn.TransformerEncoder(encoder_layer=self.transformer_layer,num_layers=2)
    self.input_proj = nn.Linear(input_size, 32)
    self.fc = nn.Linear(32, 1)
    self.sigmoid = nn.Sigmoid()
    self.pos_encoding = PositionalEncoding(32)

  def forward(self,x):

    x = self.input_proj(x)
    x = self.pos_encoding(x)
    output = self.transformer(x)
    output=self.transformer(x)
    out = output[:,-1,:]
    out = self.fc(out)
    return self.sigmoid(out)

In [ ]:
epochs = 100
batch_size = 32

for epoch in range(epochs):
    model.train()
    for i in range(0, len(x_train_t), batch_size):
        x_batch = x_train_t[i:i+batch_size]
        y_batch = y_train_t[i:i+batch_size]
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0:
      print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

Epoch 0, Loss: 0.0661
Epoch 10, Loss: 0.0416
Epoch 20, Loss: 0.0905
Epoch 30, Loss: 0.0203
Epoch 40, Loss: 0.0250
Epoch 50, Loss: 0.0279
Epoch 60, Loss: 0.2181
Epoch 70, Loss: 0.1260
Epoch 80, Loss: 0.0159
Epoch 90, Loss: 0.2614


In [ ]:
model.eval()

with torch.no_grad():
  y_pred_train = model(x_train_t)
  y_pred_train_class = (y_pred_train > 0.5).float()
  train_accuracy = (y_pred_train_class == y_train_t).float().mean()
  print(f'Train Accuracy: {train_accuracy.item():.4f}')

Train Accuracy: 0.9836


In [ ]:
model.eval()

with torch.no_grad():
  y_pred_test = model(x_test_t)
  y_pred_test_class = (y_pred_test > 0.5).float()
  test_accuracy = (y_pred_test_class == y_test_t).float().mean()
  print(f'Train Accuracy: {test_accuracy.item():.4f}')

Train Accuracy: 0.5288
